[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/10_pandas_sql/10_2_SQLite_Setup.ipynb)

# 10.2: Setting Up SQLite: Your First SQL Queries

In notebook 10.1, `df.query()` filtered a DataFrame that was already loaded in memory. But most data in the world does not live in a Python variable. It lives in a **database**, a structured system for storing and querying large amounts of data that can be shared across applications, teams, and servers. SQL (Structured Query Language) is the language that almost every database understands. If you can write SQL, you can retrieve data from nearly any data system you will encounter in a professional setting.

This notebook walks through the mechanical setup, loading the Gapminder data into a lightweight database, writing the first SQL statements, and bringing the results back into pandas, and introduces the core SELECT vocabulary you will build on across the rest of the module.

In [ ]:
import pandas as pd
import sqlite3

df = pd.read_parquet("gapminder.parquet")

# Split into two relational tables
countries = df[["country", "continent"]].drop_duplicates().reset_index(drop=True)
measurements = df[["country", "year", "lifeExp", "pop", "gdpPercap"]].copy()

conn = sqlite3.connect(":memory:")
countries.to_sql("countries", conn, index=False, if_exists="replace")
measurements.to_sql("measurements", conn, index=False, if_exists="replace")

print(f"countries table: {countries.shape}")
print(f"measurements table: {measurements.shape}")

The original Gapminder dataset has continent in every row. Here it has been split into two tables: `countries` holds the mapping from country to continent (142 rows, one per country), and `measurements` holds the year-by-year observations (1,704 rows). This split is deliberate. It creates a realistic relational structure where the two tables need to be combined, which is exactly what the JOIN lesson in notebook 10.5 will cover. For now, we will work mostly with `measurements` and `countries` separately.

## Loading data into a SQLite database

SQLite is a database engine that runs entirely inside your Python process. It needs no server, no installation beyond what is already in the standard library, and no configuration. The database can live in a file on disk or in memory (`:memory:`). For learning purposes, in-memory is ideal: it is fast, clean, and disappears when the session ends.

The workflow has two lines: open a connection, then write each DataFrame into the database with `to_sql()`.

In [2]:
conn = sqlite3.connect(":memory:")

countries.to_sql("countries", conn, index=False, if_exists="replace")
measurements.to_sql("measurements", conn, index=False, if_exists="replace")

print("Tables loaded into database.")

Tables loaded into database.


`sqlite3.connect(":memory:")` creates the database. `to_sql()` writes the DataFrame into it as a table with the given name. The `index=False` argument prevents pandas from writing the integer row index as a database column, since we do not need it. The `if_exists="replace"` argument means re-running the cell will overwrite the table rather than throwing an error.

This is the full setup. Everything from here on uses `conn` to query the database.

## Your first SQL query: `SELECT * FROM`

A SQL query is a string that describes what data you want. The most basic form says: select all columns from a table. The `*` means "all columns," and `FROM` names the table. The result comes back as a regular pandas DataFrame via `pd.read_sql()`.

In [3]:
pd.read_sql("SELECT * FROM countries", conn)

,country,continent
0,Afghanistan,Asia
1,Albania,Europe
2,Algeria,Africa
3,Angola,Africa
4,Argentina,Americas
...,...,...
137,Vietnam,Asia
138,West Bank and Gaza,Asia
139,"Yemen, Rep.",Asia
140,Zambia,Africa


All 142 rows of the countries table, both columns, returned as a DataFrame. The SQL keyword `SELECT` starts every query. `FROM` names the data source. `*` is a shorthand for "give me every column." In real work you will usually name specific columns instead of using `*`, because fetching every column in a large table wastes bandwidth and makes the result harder to read.

## Selecting specific columns

To select only the columns you need, list them by name after `SELECT`, separated by commas. This is the SQL equivalent of `df[["col1", "col2"]]` in pandas.

In [4]:
pd.read_sql("""
    SELECT country, year, lifeExp
    FROM measurements
""", conn).head(8)

,country,year,lifeExp
0,Afghanistan,1952,28.801
1,Afghanistan,1957,30.332
2,Afghanistan,1962,31.997
3,Afghanistan,1967,34.020
4,Afghanistan,1972,36.088
5,Afghanistan,1977,38.438
6,Afghanistan,1982,39.854
7,Afghanistan,1987,40.822


Only `country`, `year`, and `lifeExp` come back; `pop` and `gdpPercap` are left behind in the database. The triple-quoted string lets you spread a SQL query across multiple lines for readability. SQL itself does not care about whitespace or line breaks; the newlines are just for the human reading the code.

| pandas | SQL |
|---|---|
| `df[["country", "year", "lifeExp"]]` | `SELECT country, year, lifeExp FROM measurements` |

## `LIMIT`: capping the number of rows

When a table is large, it is useful to see just the first few rows to confirm the query is working before retrieving everything. `LIMIT n` at the end of a query returns at most `n` rows. This is the SQL equivalent of `.head(n)`.

In [5]:
pd.read_sql("""
    SELECT country, year, lifeExp, gdpPercap
    FROM measurements
    LIMIT 5
""", conn)

,country,year,lifeExp,gdpPercap
0,Afghanistan,1952,28.801,779.445314
1,Afghanistan,1957,30.332,820.853030
2,Afghanistan,1962,31.997,853.100710
3,Afghanistan,1967,34.020,836.197138
4,Afghanistan,1972,36.088,739.981106


Five rows, just like `df.head(5)`. In production environments where a table has millions of rows, `LIMIT` is essential; accidentally running `SELECT *` on a 50-million-row table without a limit can lock up your connection for minutes.

| pandas | SQL |
|---|---|
| `df.head(5)` | `... LIMIT 5` |

## `ORDER BY`: sorting results

`ORDER BY` sorts the result by one or more columns. By default the sort is ascending (smallest to largest). Append `DESC` for descending order. This is the SQL equivalent of `df.sort_values()`.

In [6]:
# 10 countries with the highest life expectancy in 2007 (we will add WHERE in the next notebook)
# For now, sort the full table and look at the top
pd.read_sql("""
    SELECT country, year, lifeExp
    FROM measurements
    ORDER BY lifeExp DESC
    LIMIT 10
""", conn)

,country,year,lifeExp
0,Japan,2007,82.603
1,"Hong Kong, China",2007,82.208
2,Japan,2002,82.000
3,Iceland,2007,81.757
4,Switzerland,2007,81.701
5,"Hong Kong, China",2002,81.495
6,Australia,2007,81.235
7,Spain,2007,80.941
8,Sweden,2007,80.884
9,Israel,2007,80.745


Japan appears twice at the top, once for 2007 and once for 2002, because we have not filtered to a single year yet. The query sorted all 1,704 rows by `lifeExp` descending and returned the top 10. Chaining `ORDER BY` and `LIMIT` together is a common pattern for a "top N" question.

| pandas | SQL |
|---|---|
| `df.sort_values("lifeExp", ascending=False).head(10)` | `ORDER BY lifeExp DESC LIMIT 10` |

You can sort by multiple columns. List them in order of priority, separated by commas:

In [7]:
pd.read_sql("""
    SELECT country, year, lifeExp
    FROM measurements
    ORDER BY country ASC, year DESC
    LIMIT 10
""", conn)

,country,year,lifeExp
0,Afghanistan,2007,43.828
1,Afghanistan,2002,42.129
2,Afghanistan,1997,41.763
3,Afghanistan,1992,41.674
4,Afghanistan,1987,40.822
5,Afghanistan,1982,39.854
6,Afghanistan,1977,38.438
7,Afghanistan,1972,36.088
8,Afghanistan,1967,34.020
9,Afghanistan,1962,31.997


Afghanistan's years appear in descending order (2007 first) because country sorts ascending and year sorts descending within each country. The `ASC` keyword is the default and can be omitted, but writing it explicitly makes the intent clear when mixing directions.

## `SELECT DISTINCT`: removing duplicates

Sometimes you want the unique values in a column, not all the rows. `SELECT DISTINCT col` returns each distinct value once. This is the SQL equivalent of `df["col"].unique()`.

In [8]:
# What years are in the dataset?
pd.read_sql("SELECT DISTINCT year FROM measurements ORDER BY year", conn)

,year
0,1952
1,1957
2,1962
3,1967
4,1972
5,1977
6,1982
7,1987
8,1992
9,1997


Twelve distinct years, 1952 through 2007 in five-year steps, confirming the Gapminder panel structure. `DISTINCT` works on multiple columns too: `SELECT DISTINCT continent, year` would return every unique (continent, year) combination.

| pandas | SQL |
|---|---|
| `df["year"].unique()` | `SELECT DISTINCT year FROM measurements` |
| `df[["continent", "year"]].drop_duplicates()` | `SELECT DISTINCT continent, year FROM ...` |

## Renaming columns with `AS`

SQL lets you rename any column in the output using `AS`. This is useful when the original column name is long or when you want a computed column to have a readable name. It is the SQL equivalent of `df.rename(columns=...)`.

In [9]:
pd.read_sql("""
    SELECT country,
           year,
           lifeExp AS life_expectancy,
           gdpPercap AS gdp_per_capita
    FROM measurements
    LIMIT 5
""", conn)

,country,year,life_expectancy,gdp_per_capita
0,Afghanistan,1952,28.801,779.445314
1,Afghanistan,1957,30.332,820.853030
2,Afghanistan,1962,31.997,853.100710
3,Afghanistan,1967,34.020,836.197138
4,Afghanistan,1972,36.088,739.981106


The output columns are renamed to `life_expectancy` and `gdp_per_capita`. The underlying table is unchanged; `AS` only renames columns in the result. You will see `AS` used constantly in the GROUP BY notebook, where computed aggregates like `AVG(lifeExp)` need a readable name assigned to them.

## SQL vocabulary so far

Here is the growing side-by-side translation between SQL clauses and their pandas equivalents:

| Goal | pandas | SQL |
|---|---|---|
| Select columns | `df[["a", "b"]]` | `SELECT a, b FROM table` |
| All columns | `df` | `SELECT * FROM table` |
| First N rows | `df.head(N)` | `... LIMIT N` |
| Sort ascending | `df.sort_values("col")` | `ORDER BY col ASC` |
| Sort descending | `df.sort_values("col", ascending=False)` | `ORDER BY col DESC` |
| Unique values | `df["col"].unique()` | `SELECT DISTINCT col FROM table` |
| Rename column | `df.rename(columns={"old": "new"})` | `SELECT old AS new FROM table` |

This table will grow with each notebook. By notebook 10.6 it will cover the full vocabulary of analytical SQL.

## What's next

You can now select columns, sort results, limit output, and deduplicate, but every query so far returns data from the entire table. The next step is filtering: keeping only the rows that satisfy a condition. In notebook 10.3 you will learn the SQL `WHERE` clause, and you will recognize it immediately from the `query()` syntax you saw in notebook 10.1.